# Assistente Médico no Google Colab

Notebook de uso do assistente médico com o modelo mesclado salvo no Google Drive.
O diretório do modelo precisa conter `config.json` e `model.safetensors`.
O repositório é clonado por HTTPS em uma célula separada para permitir atualização.
Ajuste `MODEL_PATH` se o Drive estiver em outro caminho.

Fluxo:
1. Montar o Google Drive
2. Clonar o repositório por HTTPS
3. Instalar dependências do projeto
4. Carregar o modelo local mesclado do Drive
5. Instanciar os usecases diretamente
6. Executar perguntas no loop interativo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!if [ -d /content/tech-challenge-fase-3/.git ]; then git -C /content/tech-challenge-fase-3 pull; else git clone https://github.com/JeffersonPantoja/tech-challenge-fase3.git /content/tech-challenge-fase-3; fi


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/tech-challenge-fase-3')
MODEL_PATH = Path('/content/drive/MyDrive/teach-chalenge3/medqa-finetuned-model')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

assert PROJECT_ROOT.exists(), f'Projeto não encontrado em {PROJECT_ROOT}'
assert MODEL_PATH.exists(), f'Modelo não encontrado em {MODEL_PATH}'
assert (MODEL_PATH / 'config.json').exists(), 'config.json não encontrado no modelo'
assert (MODEL_PATH / 'model.safetensors').exists(), 'model.safetensors não encontrado no modelo'

print('Projeto:', PROJECT_ROOT)
print('Modelo:', MODEL_PATH)


In [ ]:
!pip -q install --upgrade langchain transformers peft accelerate bitsandbytes sentencepiece python-dotenv


In [ ]:
from src.application.AskMedicalAssistantUseCase import AskMedicalAssistantUseCase
from src.application.LoadMedicalAssistantUseCase import LoadMedicalAssistantUseCase
from src.domain.MedicalAssistantCommandOptions import MedicalAssistantCommandOptions
from src.infrastructure.LangChainMedicalAssistantResponseGenerator import LangChainMedicalAssistantResponseGenerator
from src.infrastructure.LocalMedicalAssistantRuntimeLoader import LocalMedicalAssistantRuntimeLoader

options = MedicalAssistantCommandOptions(model_dir=MODEL_PATH)
runtime = LoadMedicalAssistantUseCase(LocalMedicalAssistantRuntimeLoader()).execute(options)
responder = LangChainMedicalAssistantResponseGenerator(runtime)
ask_use_case = AskMedicalAssistantUseCase(responder)

print(f'Modelo carregado em: {runtime.model_dir}')
print(f'LLM LangChain: {runtime.pipeline_type}')
print(f'Tokenizer: {runtime.tokenizer_name}')


In [ ]:
def ask(question: str) -> str:
    return ask_use_case.execute(question)

def chat() -> None:
    print("Digite uma pergunta ou 'sair' para encerrar.")
    while True:
        try:
            question = input('Pergunta: ').strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break

        if question.lower() in {'sair', 'exit', 'quit'}:
            break
        if not question:
            continue

        answer = ask(question)
        print(f'Resposta: {answer}')

chat()
